# 06. PDF 파서 3종 + 기본 OCR — 표는 어디서 깨지나

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/acho98/gunyang-ai-api/blob/main/labs/06_pdf_parsers_ocr.ipynb)

> 🧭 **스토리** — 규정집 PDF를 AI에게 읽히려면 먼저 **글자와 표를 꺼내야** 한다. 인사팀이 준 파일은 두 가지다. 워드에서 저장한 **디지털 PDF**와, 종이를 스캔한 **이미지**. 같은 문서인데 꺼내는 방법도, 깨지는 곳도 다르다.
>
> 📖 **이론 연결** — 이론2 p5~24 (PDF 구조·파서 비교·표 추출·OCR 원리·OCR의 한계) · ⏱ **14분** · 🖥 **GPU 권장**
>
> GPU(T4)가 있으면 빠릅니다. 없어도 CPU로 돌아갑니다(느림).
> 💡 **앞 노트북을 열어 둔 상태라면** 그 노트북에서 `런타임 → 런타임 연결 해제 및 삭제`를 먼저 하세요. 무료 코랩은 GPU 세션을 하나만 허용합니다.

**셀 배지**: ⚙️ 준비 · 🔵 따라 실행 · 🟡 바꿔서 실행 · 🤔 생각해보기 · 🟠 심화(시간 남을 때)

## 0. 준비 ⚙️
파서 3종(pdfplumber · PyMuPDF/pymupdf4llm · opendataloader-pdf)과 EasyOCR을 설치합니다. EasyOCR은 첫 실행 때 모델(검출 + 한국어 인식, 약 100MB)을 받습니다. 설치 중 빨간 `ERROR: pip's dependency resolver …` 문구가 보여도, 코랩에 미리 깔린 다른 패키지(gradio · google-colab 등)와 버전이 달라 나오는 안내라 실습에는 영향이 없습니다.

In [ ]:
%pip -q install pdfplumber pymupdf pymupdf4llm opendataloader-pdf easyocr

In [ ]:
# ⚙️ GPU 확인 — T4가 잡혔는지, 메모리가 얼마인지
import torch, subprocess
HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    name = torch.cuda.get_device_name(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {name} · 메모리 {total_gb:.1f} GB")
else:
    print("GPU 없음 — CPU로 진행합니다(느립니다). 런타임 유형에서 T4를 선택하면 빨라집니다.")
DEVICE = "cuda" if HAS_GPU else "cpu"
DTYPE = torch.float16 if HAS_GPU else torch.float32   # T4는 bf16을 지원하지 않아 fp16을 씁니다

In [ ]:
# ⚙️ 실습 데이터 로더 — 깃허브(gunyang-ai-api/data)에서 자동으로 받아옵니다. 업로드 불필요.
import os, json, urllib.request
from pathlib import Path
DATA_BASE = os.environ.get("GENAI_BASE", "https://raw.githubusercontent.com/acho98/gunyang-ai-api/main/data/")
CACHE = Path("genai_data"); CACHE.mkdir(exist_ok=True)

def fetch(name: str) -> str:
    """파일 이름(예: 'hanbit_rules.pdf', 'scan/p2.png') → 로컬 경로. URL이면 내려받아 캐시."""
    if DATA_BASE.startswith("http"):
        dst = CACHE / name
        if not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(DATA_BASE + name, dst)
        return str(dst)
    return str(Path(DATA_BASE) / name)

def load_json(name: str):
    with open(fetch(name), encoding="utf-8") as f:
        return json.load(f)

def load_text(name: str) -> str:
    with open(fetch(name), encoding="utf-8") as f:
        return f.read()

# 한글 폰트(차트용) — gunyang-data의 BMDOHYEON.ttf 를 자동으로 받아 적용. 실패해도 진행.
def set_korean_font():
    import matplotlib.pyplot as plt
    from matplotlib import font_manager as fm
    font_url = "https://raw.githubusercontent.com/acho98/gunyang-data/main/BMDOHYEON.ttf"
    dst = CACHE / "BMDOHYEON.ttf"
    try:
        if not dst.exists():
            local = Path(DATA_BASE).parent / "BMDOHYEON.ttf" if not DATA_BASE.startswith("http") else None
            if local and local.exists():
                dst.write_bytes(local.read_bytes())
            else:
                urllib.request.urlretrieve(font_url, dst)
        fm.fontManager.addfont(str(dst))
        plt.rcParams["font.family"] = fm.FontProperties(fname=str(dst)).get_name()
        plt.rcParams["axes.unicode_minus"] = False
    except Exception as e:
        print("(안내) 한글 폰트 설정 실패 — 영문 라벨은 정상:", e)
set_korean_font()
print("데이터 소스:", DATA_BASE)

In [ ]:
# ⚙️ 채점 도구 — 파서가 뽑은 표를 정답 CSV와 셀 단위로 비교합니다
import re, pandas as pd

def norm(x):
    """공백·굵게 표시(**)를 지우고 비교. '일비 (원/일)'과 '일비(원/일)'은 같은 셀로 봅니다."""
    return re.sub(r"\s+|\*\*", "", "" if x is None else str(x))

def gt_rows(name):
    """정답 CSV → [머리글, 1행, 2행, …] (문자열 리스트의 리스트)"""
    df = pd.read_csv(fetch(f"tables/{name}.csv"), dtype=str).fillna("")
    return [list(df.columns)] + df.values.tolist()

def match_rate(name, rows):
    """정답의 각 행마다 가장 비슷한 추출 행을 찾아, 같은 열 위치에 같은 값이 나온 셀의 비율(0~1)."""
    gt = [[norm(c) for c in r] for r in gt_rows(name)]
    rows = [[norm(c) for c in r] for r in (rows or [])]
    if not rows:
        return 0.0
    hit = sum(max(sum(1 for j, c in enumerate(g) if j < len(r) and r[j] == c) for r in rows) for g in gt)
    return hit / sum(len(g) for g in gt)

def md_tables(md):
    """마크다운 문자열 안의 | 표 | 들을 [[셀, …], …] 리스트로 꺼냅니다(구분선 |---| 은 제외)."""
    tabs, cur = [], []
    for line in md.splitlines() + [""]:
        s = line.strip()
        if s.startswith("|") and s.endswith("|"):
            cells = [c.strip() for c in s.strip("|").split("|")]
            if not all(re.fullmatch(r":?-{3,}:?", c) for c in cells if c):
                cur.append(cells)
        elif cur:
            tabs.append(cur); cur = []
    return tabs

def table_after_caption(md, caption):
    """'표 1.' 같은 캡션 줄 다음부터 다음 캡션·제목 전까지에서 첫 마크다운 표를 찾습니다."""
    lines = md.splitlines()
    idx = [i for i, l in enumerate(lines) if caption in l]
    if not idx:
        return []
    seg = []
    for l in lines[idx[0] + 1:]:
        if re.search(r"표 \d+\.", l) or l.lstrip().startswith("#"):
            break
        seg.append(l)
    tabs = md_tables("\n".join(seg))
    return tabs[0] if tabs else []

PDF = fetch("hanbit_rules.pdf")
print("정답 표 1(국내 출장비):"); display(pd.DataFrame(gt_rows("t1_domestic")[1:], columns=gt_rows("t1_domestic")[0]))

## 1. pdfplumber — 선이 있는 표는 잘 잡는다 🔵
`pdfplumber`(MIT)는 PDF 안의 글자 좌표와 **선(line)** 을 읽어 표를 찾습니다. 2쪽(국내 출장, 괘선 있는 표 1)과 3쪽(해외 출장, **괘선 없는** 표 2)을 차례로 봅니다. 쪽 번호는 0부터 셉니다(`pages[1]` = 2쪽).

In [ ]:
import pdfplumber
results = {}   # (파서, 표) → 셀 일치율

with pdfplumber.open(PDF) as pdf:
    print(f"총 {len(pdf.pages)}쪽\n--- 2쪽 본문 앞부분 ---")
    print(pdf.pages[1].extract_text()[:260], "…")
    for page_no, name in [(1, "t1_domestic"), (2, "t2_overseas")]:
        tables = pdf.pages[page_no].extract_tables()
        print(f"\n{page_no+1}쪽: 찾은 표 {len(tables)}개")
        if tables:
            display(pd.DataFrame(tables[0][1:], columns=tables[0][0]))
        results[("pdfplumber", name)] = max([match_rate(name, t) for t in tables], default=0.0)
print({k[1]: f"{v:.0%}" for k, v in results.items()})

💡 **해석** — 강사 환경에서 괘선 있는 표 1은 **100%** 정답과 같았고, 괘선 없는 표 2는 **표를 0개** 찾았습니다(일치율 0%). 본문 줄바꿈도 PDF에 찍힌 위치 그대로 들어옵니다. pdfplumber는 "선으로 둘러싼 칸"을 표로 보기 때문에, 선을 빼고 여백만으로 정렬한 표는 표로 인식하지 못합니다. **디지털 PDF + 괘선 표**라면 무료 파서로 충분하다는 것이 첫 번째 교훈입니다(이론2 p14).

## 2. PyMuPDF · pymupdf4llm — 표 찾기와 마크다운 변환 🔵
PyMuPDF는 빠른 PDF 엔진이고, `find_tables()`로 표를 DataFrame으로 꺼냅니다. `pymupdf4llm`은 PDF를 **LLM용 마크다운**으로 한 번에 바꿉니다.
> ⚖️ PyMuPDF·pymupdf4llm은 **AGPL** 라이선스입니다. 사내 서비스에 넣으려면 소스 공개 의무나 상용 라이선스를 검토해야 합니다.

In [ ]:
import pymupdf
doc = pymupdf.open(PDF)
for page_no, name in [(1, "t1_domestic"), (2, "t2_overseas")]:
    found = doc[page_no].find_tables().tables
    print(f"{page_no+1}쪽: 찾은 표 {len(found)}개")
    dfs = [t.to_pandas() for t in found]
    if dfs:
        display(dfs[0])
    results[("PyMuPDF find_tables", name)] = max([match_rate(name, [list(d.columns)] + d.values.tolist()) for d in dfs], default=0.0)

In [ ]:
import pymupdf4llm
md_4llm = pymupdf4llm.to_markdown(PDF, pages=[1, 2])      # 2·3쪽만 마크다운으로
print(md_4llm[-900:])                                      # 뒷부분(표 2 근처)만 출력
for name, cap in [("t1_domestic", "표 1."), ("t2_overseas", "표 2.")]:
    results[("pymupdf4llm", name)] = match_rate(name, table_after_caption(md_4llm, cap))
# pymupdf4llm은 불러오는 순간 PyMuPDF를 '레이아웃 모드'로 바꿉니다. 위 find_tables 셀을 다시 돌려도 같은 결과가 나오도록 원래 모드로 되돌립니다.
pymupdf4llm.use_layout(False)

💡 **해석** — `find_tables()`도 표 1은 완벽하지만 표 2는 0개입니다(같은 '선 기반' 방식). 반면 `pymupdf4llm`은 레이아웃 모델(pymupdf-layout)을 함께 써서 **표 2의 본문 5행을 표로 살려 냅니다.** 단, 출력을 보면 머리글 행(`지역 | 일비 (과장 이하) | …`)이 사라졌습니다. 숫자는 있는데 **어느 열이 무엇인지 모르는 표**가 된 것이죠. RAG에서 "차장 이상 북미 일비?"를 물으면 틀릴 수 있는 상태입니다.

## 3. opendataloader-pdf — 읽기 순서 + JSON 🔵
`opendataloader-pdf`(Apache 2.0)는 XY-Cut++ 읽기 순서와 요소별 JSON(제목·문단·표·좌표)을 함께 줍니다. **Java 11 이상**이 필요합니다. 코랩에 Java가 없으면 설치를 시도하고(약 1분), 실패하면 이 셀은 건너뜁니다.

In [ ]:
import shutil, subprocess, glob
HAS_JAVA = shutil.which("java") is not None
if not HAS_JAVA and os.path.exists("/content"):          # 코랩에서만 자동 설치
    print("Java가 없어 설치합니다(약 1분)…")
    subprocess.run("apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless",
                   shell=True, capture_output=True, text=True)
    HAS_JAVA = shutil.which("java") is not None

md_odl = ""
if HAS_JAVA:
    try:
        import opendataloader_pdf
        opendataloader_pdf.convert(input_path=[PDF], output_dir="odl_out", format="markdown,json")  # 로그가 많이 찍혀도 정상
        md_odl = open(glob.glob("odl_out/*.md")[0], encoding="utf-8").read()
        odl_json = json.load(open(glob.glob("odl_out/*.json")[0], encoding="utf-8"))
        print("JSON 요소 종류:", sorted({k["type"] for k in odl_json["kids"]}), "· 요소 수", len(odl_json["kids"]))
        i = md_odl.find("표 2.")
        print("--- 표 2 주변 마크다운 ---\n" + md_odl[i - 2:i + 260])
    except Exception as e:
        print("opendataloader 실행 실패 — 건너뜀:", type(e).__name__, str(e)[:200])
else:
    print("Java 없음 — opendataloader 셀은 건너뜀")
if md_odl:
    for name, cap in [("t1_domestic", "표 1."), ("t2_overseas", "표 2.")]:
        results[("opendataloader", name)] = match_rate(name, table_after_caption(md_odl, cap))

💡 **해석** — 표 1은 역시 100%입니다. 표 2는 `- 표 2. 해외 출장비 … 지역 일비 (과장 이하) … 북미 60 80 220 유럽 …`처럼 **한 줄 문장으로 펴져** 버렸습니다(일치율 0%). 글자는 하나도 빠지지 않았지만 행·열 구조가 사라졌습니다. 그래도 JSON에 제목·목록·표가 요소별로 나뉘어 있어 **청킹 경계**를 잡기에는 좋습니다.

## 4. 한눈에 비교 — 괘선 표 vs 무괘선 표 🔵

In [ ]:
cmp = (pd.Series(results).unstack().rename(columns={"t1_domestic": "표1 국내(괘선)", "t2_overseas": "표2 해외(무괘선)"}))
display((cmp * 100).round(0).astype(int).astype(str) + "%")

💡 **해석** — 강사 환경 결과입니다.

| 파서 | 표1 (괘선) | 표2 (무괘선) | 표2에서 무슨 일이? |
|---|---|---|---|
| pdfplumber | 100% | 0% | 표를 못 찾음 |
| PyMuPDF `find_tables` | 100% | 0% | 표를 못 찾음 |
| pymupdf4llm | 100% | 83% | 본문 5행은 살리고 **머리글 행 소실** |
| opendataloader | 100% | 0% | 한 줄 문장으로 펴짐 |

**괘선 있는 디지털 표는 어떤 파서든 공짜로 100%**입니다. 무괘선 표처럼 "사람 눈에는 표인데 선이 없는" 경우에서 차이가 납니다. 개발사가 공개한 벤치마크(이론2 p12)보다 **내 문서로 직접 재 보는 것**이 정답입니다.

## 5. 유니코드 정규화(NFC) — 눈에 같은데 검색이 안 된다 🔵
macOS에서 만든 파일이나 일부 시스템 export는 한글을 **자모 분리형(NFD)** 으로 저장합니다. 같은 규정집을 NFD로 저장한 파일에서 "출장"을 검색해 봅니다.

In [ ]:
import unicodedata
nfd_text = load_text("hanbit_rules_gt_nfd.md")
print("화면에는 똑같이 보임:", nfd_text[:30])
print('"출장" in 원본(NFD)      →', "출장" in nfd_text, f"(글자 수 {len(nfd_text):,})")
fixed = unicodedata.normalize("NFC", nfd_text)
print('"출장" in NFC 정규화 후  →', "출장" in fixed, f"(글자 수 {len(fixed):,}) · 등장 {fixed.count('출장')}회")

💡 **해석** — NFD는 '출'을 `ㅊ+ㅜ+ㄹ` 세 코드로 저장합니다. 그래서 키보드로 친 "출장"(NFC)과 **바이트가 달라** `in` 검색, BM25, 임베딩 모두 엉뚱하게 동작합니다. 파서 출력이든 OCR 출력이든 **저장 전에 `unicodedata.normalize("NFC", s)` 한 줄**을 넣으세요(노트북 02 4절, 이론1 p127).

## 6. 스캔본은 OCR — EasyOCR 🔵
스캔본(`scan/p2.png`)은 글자가 아니라 **그림**입니다. PDF 파서로는 아무것도 못 꺼냅니다. EasyOCR은 **글자 영역 검출(CRAFT) → 글자 인식** 두 단계로 `(박스 4점, 글자, 신뢰도)` 목록을 돌려줍니다(이론2 p19).

In [ ]:
import easyocr, time
t0 = time.time()
reader = easyocr.Reader(["ko", "en"], gpu=HAS_GPU)        # 첫 실행 때 모델 다운로드(약 100MB)
SCAN = fetch("scan/p2.png")
ocr = reader.readtext(SCAN)                               # [(bbox 4점, text, conf), …]
print(f"검출 {len(ocr)}개 · {time.time()-t0:.1f}초 ({DEVICE})\n")
for bbox, text, conf in ocr[:12]:
    x, y = map(int, bbox[0])
    print(f"({x:4d},{y:4d})  conf={conf:.2f}  {text}")

💡 **해석** — 강사 환경(Mac CPU)에서 2쪽 한 장에 **40개 박스**, 시간은 10초~1분대(다른 작업 부하에 따라)였습니다. 코랩 T4에서는 약 10초였습니다(2026-10-17 실측 9.8초, 모델 다운로드 제외). 본문 줄을 보면 `구성한다 → 구성하다`, `택시를 → 택시틀`, `을 → 올`처럼 **받침 하나가 틀린 오류**가 많고 신뢰도도 0.3~0.5로 낮습니다. 반면 `사원`, `20,000` 같은 짧은 표 셀은 신뢰도 0.7~1.0으로 높습니다. 회전(±1.5°)과 노이즈가 섞인 스캔본의 현실입니다.

In [ ]:
from PIL import Image, ImageDraw
img = Image.open(SCAN).convert("RGB")
draw = ImageDraw.Draw(img)
for bbox, text, conf in ocr:
    color = (0, 160, 0) if conf >= 0.7 else (230, 140, 0) if conf >= 0.4 else (220, 0, 0)   # 초록=높음, 주황, 빨강=낮음
    draw.polygon([tuple(map(int, p)) for p in bbox], outline=color, width=4)
display(img.crop((0, 0, img.width, 1150)).resize((img.width // 2, 575)))   # 위쪽 절반만 축소해서 표시

In [ ]:
para = reader.readtext(SCAN, detail=0, paragraph=True)   # 박스를 문단으로 합치고 글자만
print(f"문단 {len(para)}개")
for p in para:
    print("-", p[:300], "…" if len(p) > 300 else "")

💡 **해석** — `paragraph=True`는 가까운 박스를 문단으로 합쳐 줍니다. 그런데 결과를 보면 본문과 **표 1이 한 문단으로 뭉쳐** `직급 일비 (원 일) … 사원 20,000 30,000 80,000 대리 …`처럼 숫자가 줄줄이 이어집니다. OCR은 **글자는 주지만 구조(문단·표·읽기 순서)는 주지 않습니다**(이론2 p22). 그리고 `원/일`의 `/`가 사라진 것도 보이나요?

## 7. 표 영역만 — 행·열을 되살릴 수 있을까? 🔵
OCR 결과에서 "표 1" 캡션의 y좌표를 찾아 그 아래 박스만 모은 뒤, **y좌표가 비슷한 박스끼리 한 행**으로 묶어 봅니다. 묶는 허용 오차(`tol`, 픽셀)를 바꾸면 어떻게 될까요?

In [ ]:
cap_y = next(b[0][1] for b, t, c in ocr if t.replace(" ", "").startswith("표1"))
cells = [(b, t, c) for b, t, c in ocr if b[0][1] > cap_y + 20]          # 캡션 아래 = 표 영역
display(img.crop((100, int(cap_y) - 10, 1550, 1090)).resize((725, 240)))

def rows_by_y(items, tol):
    """y가 행의 첫 박스와 tol 픽셀 이내면 같은 행. 행 안에서는 x 순서로 정렬."""
    rows, start = [], None
    for b, t, c in sorted(items, key=lambda x: x[0][0][1]):
        y = b[0][1]
        if start is None or y - start > tol:
            rows.append([]); start = y
        rows[-1].append((b[0][0], t))
    return [[t for x, t in sorted(r)] for r in rows]

for tol in [15, 30, 60]:
    rows = rows_by_y(cells, tol)
    print(f"tol={tol:>2}px → {len(rows):>2}행 · 셀 일치율 {match_rate('t1_domestic', rows):.0%}")
print("\n[tol=15 결과 앞 6행]")
display(pd.DataFrame(rows_by_y(cells, 15)).head(6))

💡 **해석** — 강사 환경 결과: `tol=15` → 14행·68%, `tol=30` → 7행·**89%**, `tol=60` → 5행·39%.
- 오차가 **작으면** 스캔이 1.5° 기울어 같은 행의 오른쪽 칸(y가 20px 더 큼)이 **다음 행으로 떨어집니다**(위 표에서 `80,000`이 혼자 한 행).
- 오차가 **크면** 위아래 행이 **한 행으로 합쳐집니다**.
- 딱 맞는 값(30)에서도 머리글의 `원/일`이 `원 일`로 읽혀 100%가 안 됩니다.

이런 수작업 규칙은 페이지마다 깨집니다. 그래서 **레이아웃 모델로 "여기가 표"를 먼저 찾고, 표 구조 모델로 칸을 나누는** 다음 층이 필요합니다(노트북 07·08).

In [ ]:
# 신뢰도(conf) 분포 — 임계값으로 거르면 무엇이 남고 무엇이 버려질까?
confs = pd.Series([c for _, _, c in ocr])
for th in [0.3, 0.5, 0.7]:
    print(f"conf ≥ {th}: {int((confs >= th).sum()):>2}/{len(confs)}개 남음")
print("\n버려지는 박스(conf < 0.5) 예:")
for b, t, c in ocr:
    if c < 0.5:
        print(f"  {c:.2f}  {t[:45]}")

💡 **해석** — 임계값을 0.5로 올리면 표 숫자는 대부분 남지만 **본문 문장 줄이 통째로 버려집니다**. 한국어 긴 줄은 한두 글자만 틀려도 줄 전체 신뢰도가 낮게 나오기 때문입니다. 신뢰도는 "버릴지" 보다 **"사람이 검수할 곳"을 고르는 데** 쓰는 것이 실무적입니다.

## 8. 바꿔서 실행 🟡
1. **다른 표로 재비교**: `PAGE_NO, TABLE, CAPTION`을 4쪽 경조사 휴가 표(`3, "t4_family_leave", "표 4."`)로 두고 실행하세요. 같은 쪽에 표가 2개(표 3·4)일 때 파서들이 어떻게 하나요?
2. **다른 스캔 + 임계값**: `SCAN2`를 `scan/p4.png`(표 2개)로, `CONF_TH`를 0.3·0.5·0.8로 바꿔 보세요.

In [ ]:
PAGE_NO, TABLE, CAPTION = 3, "t4_family_leave", "표 4."     # ← 바꿔 보세요 (0부터 센 쪽 번호)
r2 = {}
with pdfplumber.open(PDF) as pdf:
    tabs = pdf.pages[PAGE_NO].extract_tables()
r2["pdfplumber"] = max([match_rate(TABLE, t) for t in tabs], default=0.0)
dfs = [t.to_pandas() for t in pymupdf.open(PDF)[PAGE_NO].find_tables().tables]
r2["PyMuPDF find_tables"] = max([match_rate(TABLE, [list(d.columns)] + d.values.tolist()) for d in dfs], default=0.0)
r2["pymupdf4llm"] = match_rate(TABLE, table_after_caption(pymupdf4llm.to_markdown(PDF, pages=[PAGE_NO]), CAPTION))
pymupdf4llm.use_layout(False)
if md_odl:
    r2["opendataloader"] = match_rate(TABLE, table_after_caption(md_odl, CAPTION))
print(f"{PAGE_NO+1}쪽에서 찾은 표 수 — pdfplumber {len(tabs)}개 · PyMuPDF {len(dfs)}개")
display(pd.Series(r2, name=TABLE).map("{:.0%}".format).to_frame())

In [ ]:
SCAN2, CONF_TH = "scan/p4.png", 0.5                     # ← 바꿔 보세요
ocr2 = reader.readtext(fetch(SCAN2))
kept = [(b, t, c) for b, t, c in ocr2 if c >= CONF_TH]
print(f"{SCAN2}: 검출 {len(ocr2)}개 → conf ≥ {CONF_TH} 남은 {len(kept)}개")
print("남은 박스 앞 15개:", [t for _, t, _ in kept[:15]])

### 🤔 생각해보기

이 규정집이라면 **어느 층에서 멈춰도 될까요?** 디지털 PDF일 때와 스캔본일 때를 나눠 답해 보세요. (층: ① 기본 파서 → ② OCR → ③ 레이아웃·표 구조 → ④ VLM)

<details><summary><b>🤔 생각해보기 — 해설 보기(클릭)</b></summary>

- **디지털 PDF**: 표 대부분이 괘선 표이므로 ① 기본 파서로 충분합니다(비용 0, 수 초). 단, 무괘선 표 2처럼 깨지는 표가 **하나라도** 있으면 그 쪽만 ③(Docling 등)으로 보내거나 사람이 검수합니다. "전체를 무거운 도구로"가 아니라 "깨지는 곳만 위 층으로"가 비용 효율적입니다.
- **스캔본**: ②만으로는 표 구조가 사라지고(행·열 섞임) 받침 오류가 많습니다. 최소 ③(레이아웃 + 표 구조)이 필요하고, 한국어 품질이 중요하면 ④ 한국어를 지원하는 VLM(PaddleOCR-VL)이나 상용 OCR을 검토합니다.
- 어느 경우든 **정답 몇 개를 만들어 셀 일치율로 재 보는 것**이 선택 근거가 됩니다(이론2 p47·p55).

</details>

---
### ✅ 이 노트북에서 가져갈 것
- 괘선 있는 디지털 표는 pdfplumber·PyMuPDF·opendataloader 모두 100%. 차이는 무괘선 표에서 난다.
- 파서는 글자를 꺼내도 구조(머리글·행·열)를 잃을 수 있다. 정답 CSV로 셀 일치율을 재라.
- 추출 텍스트는 저장 전에 NFC 정규화. NFD면 검색이 실패한다.
- OCR은 (박스, 글자, 신뢰도)만 준다. 읽기 순서·표 구조는 다음 층(레이아웃·VLM)의 몫.

> 다음 노트북: **07_layout_vlm_parsing — 레이아웃 모델로 구조를 보고, VLM이 통째로 읽는다**